01. RunnableWithMessageHistory에 ChatMessageHistory추가

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

# ChatOllama 멀티모달 언어 모델을 불러옵니다.
#--------------------------------
# LLM
#--------------------------------
from langchain_ollama import ChatOllama

# Model
LLM_MODEL = "gemma3:1b"
llm = ChatOllama(
        model = LLM_MODEL,
        temperature=0.1,
        base_url="http://localhost:11434",
        top_k=3
    )

# 프롬프트 정의
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 Question-Answering 챗봇입니다. 주어진 질문에 대한 답변을 제공해주세요.",
        ),
        # 대화기록용 key 인 chat_history 는 가급적 변경 없이 사용하세요!
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "#Question:\n{question}"),  # 사용자 입력을 변수로 사용
    ]
)

# 일반 Chain 생성
chain = prompt | llm | StrOutputParser()



In [ ]:
# 세션 기록을 저장할 딕셔너리
store = {}


# 세션 ID를 기반으로 세션 기록을 가져오는 함수
def get_session_history(session_ids):
    print(f"[대화 세션ID]: {session_ids}")
    if session_ids not in store:  # 세션 ID가 store에 없는 경우
        # 새로운 ChatMessageHistory 객체를 생성하여 store에 저장
        store[session_ids] = ChatMessageHistory()
    return store[session_ids]  # 해당 세션 ID에 대한 세션 기록 반환

In [ ]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,  # 세션 기록을 가져오는 함수
    input_messages_key = "question",  # 사용자의 질문이 템플릿 변수에 들어갈 key
    history_messages_key = "chat_history",  # 기록 메시지의 키
)

In [ ]:
# 두번째 질문
result = chain_with_history.invoke(
    # 질문 입력
    {"question": "내 이름이 뭐라고?"},
    # 세션 ID 기준으로 대화를 기록합니다.
    config={"configurable": {"session_id": "abc123"}},
)

for chunk in result:
    print(chunk, end="", flush=True)

02. SQLite 에 대화내용 저장 (SQLAlchemy 사용)

In [ ]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

# SQLChatMessageHistory 객체를 생성하고 세션 ID와 데이터베이스 연결 파일을 설정
chat_message_history = SQLChatMessageHistory(
    session_id="sql_history", connection="sqlite:///sqlite.db"
)

# 사용자 메시지를 추가합니다.
chat_message_history.add_user_message(
    "안녕? 만나서 반가워. 내 이름은 forggy야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!"
)
# AI 메시지를 추가합니다.
chat_message_history.add_ai_message("안녕 forggy, 만나서 반가워. 나도 잘 부탁해!")

In [ ]:
# 저장된 대화내용을 확인
chat_message_history.messages

In [31]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

In [33]:
#--------------------------------
# LLM
#--------------------------------
from langchain_ollama import ChatOllama

# Model
LLM_MODEL = "gemma3:1b"
llm = ChatOllama(
        model = LLM_MODEL,
        temperature=0.1,
        base_url="http://localhost:11434",
        top_k=3
    )

# Prompt 생성
prompt = ChatPromptTemplate.from_messages(
    [
        # 시스템 메시지
        ("system", "너는 훌륭한 AI 조력자야."),
        # 대화 기록을 위한 Placeholder
        MessagesPlaceholder(variable_name = "chat_history"),
        ("human", "{question}"),  # 질문
    ]
)

# Chain 생성
chain = prompt | llm | StrOutputParser()

# SQLLite db에서 대화내용 조회 함수
def get_chat_history(user_id, conversation_id):
    return SQLChatMessageHistory(
        table_name = user_id,
        session_id = conversation_id,
        connection="sqlite:///sqlite.db",
    )

In [36]:
from langchain_core.runnables.utils import ConfigurableFieldSpec

config_fields = [
    ConfigurableFieldSpec(
        id = "user_id",
        annotation = str,
        name = "User ID",
        description = "Unique identifier for a user.",
        default = "",
        is_shared = True,
    ),
    ConfigurableFieldSpec(
        id = "conversation_id",
        annotation = str,
        name = "Conversation ID",
        description = "Unique identifier for a conversation.",
        default = "",
        is_shared = True,
    ),
]

In [37]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,  # 대화 기록을 가져오는 함수를 설정합니다.
    input_messages_key = "question",  # 입력 메시지의 키를 "question"으로 설정
    history_messages_key = "chat_history",  # 대화 기록 메시지의 키를 "history"로 설정
    history_factory_config = config_fields,  # 대화 기록 조회시 참고할 파라미터를 설정합니다.
)

In [38]:
# config 설정
config = {"configurable": {"user_id": "forggy", "conversation_id": "conversation_froggy"}}

In [43]:
# 질문과 config 를 전달하여 실행
response = chain_with_history.stream({"question": "안녕 반가워, 내 이름은 froggy야"}, config)
for chunk in response:
    print(chunk, end="", flush=True)

안녕하세요, froggy! 만나서 반가워요. 당신은 정말 재미있는 이름이네요! 😊 혹시 어떤 이야기를 하고 싶으신가요?

In [52]:
# 후속 질문을 실행
response = chain_with_history.stream({"question": "내 이름이 뭐라고?"}, config)
for chunk in response:
    print(chunk, end="", flush=True)

당신은 "Froggy"입니다! 😄